# 0. Kaggle Setup & GitHub Repository Cloning
Automatically clones the GitHub repo and installs required dependencies when running in Kaggle.

In [11]:
# Install required dependencies
!pip install rank-bm25 rapidfuzz -q

import sys
import os
from pathlib import Path

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/input")
print(f"Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    repo_dir = Path("/kaggle/working/repo")
    github_url = os.environ.get("GITHUB_REPO_URL", "https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026.git")
    if not repo_dir.exists():
        print("Cloning repository from GitHub...")
        !git clone {github_url} /kaggle/working/repo
    else:
        print("Pulling latest code from GitHub...")
        !git -C /kaggle/working/repo pull
    
    if repo_dir.exists():
        os.chdir(str(repo_dir))
        sys.path.insert(0, str(repo_dir))
    print(f"Current working directory: {os.getcwd()}")
else:
    project_root = Path(os.getcwd()).parent if os.path.basename(os.getcwd()) == "notebooks" else Path(os.getcwd())
    sys.path.insert(0, str(project_root))

print("Setup completed.")

Running on Kaggle: True
Pulling latest code from GitHub...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 356 bytes | 356.00 KiB/s, done.
From https://github.com/ChandrimaNandi/Amazon-ML-Hackathon-2026
   0fbe9aa..867d6ee  main       -> origin/main
Updating 0fbe9aa..867d6ee
Fast-forward
 src/ranking.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Current working directory: /kaggle/working/repo
Setup completed.


# 1. Imports and configuration

In [12]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(os.getcwd()).parent if os.path.basename(os.getcwd()) == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_S1_PATH, TRAIN_S2_PATH, TRAIN_S3_PATH, TRAIN_GROUND_TRUTH_PATH,
    TEST_S1_PATH, TEST_S2_PATH, TEST_S3_PATH, TEST_DIR,
    SUBMISSION_MATCHING_PATH, SUBMISSION_CANDIDATE_PATH, RESULTS_DIR
)
from src.data_loader import load_source_tsv, load_ground_truth
from src.profiling import profile_dataframe, profile_ground_truth, detect_script
from src.normalization import create_normalized_features
from src.retrieval import BM25Retriever, CharTFIDFRetriever
from src.candidate_generation import generate_candidate_union
from src.similarity import compute_string_similarities
from src.features import extract_candidate_features
from src.ranking import EntityMatcherModel, mine_hard_negatives
from src.thresholding import apply_decision_rules
from src.singleton import analyze_singleton_performance
from src.evaluation import evaluate_macro_metrics, evaluate_candidate_recall
from src.inference import generate_submission_files

print('Imports and configuration loaded successfully.')

Imports and configuration loaded successfully.


# 2. Dataset loading

In [13]:
print('[INFO] Loading training datasets...')
s1_df = load_source_tsv(TRAIN_S1_PATH)
s2_df = load_source_tsv(TRAIN_S2_PATH)
s3_df = load_source_tsv(TRAIN_S3_PATH)
gt_df, s1_to_matches, _ = load_ground_truth(TRAIN_GROUND_TRUTH_PATH)

print(f'S1 shape: {s1_df.shape}')
print(f'S2 shape: {s2_df.shape}')
print(f'S3 shape: {s3_df.shape}')
print(f'Ground truth shape: {gt_df.shape}')

display(s1_df.head(3))

2026-09-24 21:42:27,168 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/train/train_source1.tsv


[INFO] Loading training datasets...


2026-09-24 21:42:34,267 [INFO] Loaded 2,206,821 rows from train_source1.tsv
2026-09-24 21:42:34,268 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/train/train_source2.tsv
2026-09-24 21:42:49,448 [INFO] Loaded 5,034,616 rows from train_source2.tsv
2026-09-24 21:42:49,449 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/train/train_source3.tsv
2026-09-24 21:43:05,741 [INFO] Loaded 5,285,603 rows from train_source3.tsv
2026-09-24 21:43:05,742 [INFO] Loading Ground Truth from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/train/train_ground_truth.tsv
2026-09-24 21:44:39,798 [INFO] Loaded 2,206,821 ground truth S1 entities (7,638,365 total match mappings)


S1 shape: (2206821, 4)
S2 shape: (5034616, 4)
S3 shape: (5285603, 4)
Ground truth shape: (2206821, 2)


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


# 3. Dataset profiling

In [15]:
print('=== Train S1 Profiling ===')
profile_s1 = profile_dataframe(s1_df, 'Train S1')
print('Missing values:')
print(f'  Name missing: {profile_s1["missing_name_count"]} ({profile_s1["missing_name_pct"]}%)')
print(f'  Address missing: {profile_s1["missing_address_count"]} ({profile_s1["missing_address_pct"]}%)')
print(f'  Country missing: {profile_s1["missing_country_count"]} ({profile_s1["missing_country_pct"]}%)')

print('Country distribution:')
print(profile_s1['countries'])

=== Train S1 Profiling ===
Missing values:
  Name missing: 0 (0.0%)
  Address missing: 0 (0.0%)
  Country missing: 0 (0.0%)
Country distribution:
{'US': 1323633, 'India': 883188}


# 4. Multilingual / multiscript analysis

In [17]:
print('=== Script Analysis ===')
print('Name Scripts in Source 1:')
print(profile_s1['name_scripts'])

print('Address Scripts in Source 1:')
print(profile_s1['address_scripts'])

=== Script Analysis ===
Name Scripts in Source 1:
{'Latin': 2206821}
Address Scripts in Source 1:
{'Latin': 2206821}


# 5. Ground-truth analysis

In [18]:
print('=== Ground Truth Match Breakdown ===')
gt_stats = profile_ground_truth(gt_df, s1_to_matches)
for k, v in gt_stats.items():
    print(f'  {k}: {v}')

=== Ground Truth Match Breakdown ===
  total_s1_entities: 2206821
  zero_match_count: 123247
  zero_match_pct: 5.58
  one_match_count: 119157
  one_match_pct: 5.4
  multi_match_count: 1964417
  multi_match_pct: 89.02
  avg_matches_per_s1: 3.461
  max_matches_per_s1: 11


# 6. Exact-match baseline

In [19]:
query_df = pd.concat([s2_df, s3_df], ignore_index=True)
if len(query_df) > 10000:
    query_sample = query_df.sample(n=10000, random_state=42).reset_index(drop=True)
else:
    query_sample = query_df

s1_sample = s1_df.sample(n=min(50000, len(s1_df)), random_state=42).reset_index(drop=True)

print('Exact match baseline on sample...')
exact_names = set(s1_sample['business_name'])
exact_matches = sum(1 for n in query_sample['business_name'] if n in exact_names)
print(f'Exact Name Matches: {exact_matches:,} / {len(query_sample):,} ({exact_matches/len(query_sample)*100:.2f}%)')

Exact match baseline on sample...
Exact Name Matches: 104 / 10,000 (1.04%)


# 7. Unicode-safe normalization

In [20]:
print('Applying Unicode NFKC + casefold normalization...')
s1_norm = create_normalized_features(s1_sample)
query_norm = create_normalized_features(query_sample)

print('Sample normalized entries:')
display(s1_norm[['business_name', 'name_normalized', 'business_address', 'address_normalized']].head(3))

2026-09-24 21:47:50,900 [INFO] Applying Unicode-safe normalization to names and addresses...


Applying Unicode NFKC + casefold normalization...


2026-09-24 21:47:51,338 [INFO] Applying Unicode-safe normalization to names and addresses...


Sample normalized entries:


,business_name,name_normalized,business_address,address_normalized
0,Pediatric Medicine PLLC,pediatric medicine pllc,"4850 20, Otisco, NY",4850 20 otisco ny
1,Fetech National Twin,fetech national twin,"19034 Woodburn Road, Woodburn, IN",19034 woodburn road woodburn in
2,General Design Innovations LLC,general design innovations llc,"7241 Osage Avenue, Mesa, AZ",7241 osage avenue mesa az


# 8. BM25 retrieval

In [21]:
print('Fitting BM25 Name retriever...')
bm25 = BM25Retriever()
bm25.fit(s1_norm['name_normalized'].tolist(), s1_norm['entity_id'].tolist())

res = bm25.retrieve_top_k(query_norm['name_normalized'].head(5).tolist(), top_k=5)
for i, top_k in enumerate(res):
    print(f'Query: {query_norm["name_normalized"].iloc[i]} -> Candidates: {top_k[:3]}')

2026-09-24 21:47:57,864 [INFO] Fitting BM25 index on 50,000 texts...
2026-09-24 21:47:57,986 [INFO] BM25 fitted in 0.12s.
2026-09-24 21:47:57,991 [INFO] Retrieving top 5 candidates for 5 queries using BM25...


Fitting BM25 Name retriever...


2026-09-24 21:47:58,126 [INFO] BM25 retrieval finished in 0.13s.


Query: jhajjar pictures private limited -> Candidates: [('S1-682828443', 11.858469693441364, 1), ('S1-359033061', 10.592892184769221, 2), ('S1-488826502', 2.786881587702927, 3)]
Query: rodriguez riemning inc -> Candidates: [('S1-178360736', 9.743347411978327, 1), ('S1-612125427', 9.743347411978327, 2), ('S1-914470113', 8.627539334253806, 3)]
Query: orthopedic center ltd -> Candidates: [('S1-698377693', 12.648317240772151, 1), ('S1-240005367', 12.648317240772151, 2), ('S1-206768974', 10.932129208273018, 3)]
Query: vanguard services -> Candidates: [('S1-39299983', 8.67048661717047, 1), ('S1-626909789', 7.485300463838319, 2), ('S1-359510089', 7.485300463838319, 3)]
Query: rossetti s advanced signs llc -> Candidates: [('S1-827404843', 12.598309681061318, 1), ('S1-861534283', 11.093136485532844, 2), ('S1-635716375', 11.093136485532844, 3)]


# 9. BM25 Recall@K

In [22]:
print('Evaluating candidate generation...')
cand_df, cand_stats = generate_candidate_union(s1_norm, query_norm, k_name=20, k_address=15)
rec_stats = evaluate_candidate_recall(cand_df, s1_to_matches)

print('Candidate Recall Stats:')
for k, v in rec_stats.items():
    print(f'  {k}: {v}')

2026-09-24 21:48:01,897 [INFO] ============================================================
2026-09-24 21:48:01,898 [INFO] Starting Candidate Generation for 10,000 queries against 50,000 S1 reference records
2026-09-24 21:48:01,899 [INFO] ============================================================
2026-09-24 21:48:01,992 [INFO] Fitting BM25 index on 50,000 texts...


Evaluating candidate generation...


2026-09-24 21:48:03,184 [INFO] BM25 fitted in 1.19s.
2026-09-24 21:48:03,188 [INFO] Fitting BM25 index on 50,000 texts...
2026-09-24 21:48:03,527 [INFO] BM25 fitted in 0.34s.
2026-09-24 21:48:03,535 [INFO] Fitting CharTFIDFVectorization on 50,000 texts...
2026-09-24 21:48:05,180 [INFO] CharTFIDF fitted in 1.64s. Vocabulary size: 51,725
2026-09-24 21:48:05,181 [INFO] Fitting CharTFIDFVectorization on 50,000 texts...
2026-09-24 21:48:08,413 [INFO] CharTFIDF fitted in 3.23s. Vocabulary size: 100,216
2026-09-24 21:48:08,415 [INFO] Retrieving top 20 candidates for 10,000 queries using BM25...
2026-09-24 21:53:26,959 [INFO] BM25 retrieval finished in 318.54s.
2026-09-24 21:53:26,961 [INFO] Retrieving top 20 candidates for 10,000 queries using BM25...
2026-09-24 22:11:22,243 [INFO] BM25 retrieval finished in 1075.28s.
2026-09-24 22:11:22,244 [INFO] Retrieving top 20 candidates for 10,000 queries using Char-TFIDF...
2026-09-24 22:11:39,926 [INFO] Char-TFIDF retrieval finished in 17.68s.
2026-0

Candidate Recall Stats:
  candidate_recall: 0.0
  total_true_pairs: 7638365
  found_pairs: 147
  missed_pairs: 7638218


# 10. Character TF-IDF retrieval

In [23]:
print('Fitting Character TF-IDF retriever...')
char_retriever = CharTFIDFRetriever(ngram_range=(3, 5))
char_retriever.fit(s1_norm['name_normalized'].tolist(), s1_norm['entity_id'].tolist())

res_char = char_retriever.retrieve_top_k(query_norm['name_normalized'].head(5).tolist(), top_k=5)
for i, top_k in enumerate(res_char):
    print(f'Char-TFIDF Query: {query_norm["name_normalized"].iloc[i]} -> Candidates: {top_k[:3]}')

2026-09-24 22:12:05,567 [INFO] Fitting CharTFIDFVectorization on 50,000 texts...


Fitting Character TF-IDF retriever...


2026-09-24 22:12:07,230 [INFO] CharTFIDF fitted in 1.66s. Vocabulary size: 51,725
2026-09-24 22:12:07,233 [INFO] Retrieving top 5 candidates for 5 queries using Char-TFIDF...
2026-09-24 22:12:07,278 [INFO] Char-TFIDF retrieval finished in 0.04s.


Char-TFIDF Query: jhajjar pictures private limited -> Candidates: [('S1-682828443', 0.6739637851715088, 1), ('S1-359033061', 0.6690924763679504, 2), ('S1-714234193', 0.3217266798019409, 3)]
Char-TFIDF Query: rodriguez riemning inc -> Candidates: [('S1-178360736', 0.7118686437606812, 1), ('S1-524080391', 0.6997435092926025, 2), ('S1-708858508', 0.6899551153182983, 3)]
Char-TFIDF Query: orthopedic center ltd -> Candidates: [('S1-240005367', 0.97102952003479, 1), ('S1-698377693', 0.97102952003479, 2), ('S1-118085110', 0.9549458026885986, 3)]
Char-TFIDF Query: vanguard services -> Candidates: [('S1-55594665', 0.774552583694458, 1), ('S1-626909789', 0.774552583694458, 2), ('S1-592910122', 0.7692846059799194, 3)]
Char-TFIDF Query: rossetti s advanced signs llc -> Candidates: [('S1-740665098', 0.4724130928516388, 1), ('S1-683189165', 0.44656816124916077, 2), ('S1-305781569', 0.4418291449546814, 3)]


# 11. Candidate union

In [24]:
print(f'Total Union Candidates: {len(cand_df):,}')
print(f'Average Candidates per Query: {cand_stats["avg_candidates_per_query"]}')
display(cand_df.head(3))

Total Union Candidates: 550,136
Average Candidates per Query: 55.01


,query_id,s1_id,by_exact_name,by_exact_address,by_bm25_name,bm25_name_score,bm25_name_rank,by_bm25_combined,bm25_comb_score,bm25_comb_rank,by_char_name,char_name_score,char_name_rank,by_char_address,char_addr_score,char_addr_rank,retrieval_agreement_count,best_retrieval_rank
0,S3-792642684,S1-682828443,0,0,1,11.858470,1,0,0.0,999,1,0.673964,1,0,0.0,999,2,1
1,S3-792642684,S1-359033061,0,0,1,10.592892,2,0,0.0,999,1,0.669092,2,0,0.0,999,2,2
2,S3-792642684,S1-488826502,0,0,1,2.786882,3,0,0.0,999,0,0.000000,999,0,0.0,999,1,3


# 12. Fuzzy retrieval

In [25]:
print('Fuzzy token set ratios sample...')
for row in cand_df.head(3).itertuples():
    print(f'Query {row.query_id} vs S1 {row.s1_id}: Agreement count = {row.retrieval_agreement_count}')

Fuzzy token set ratios sample...
Query S3-792642684 vs S1 S1-682828443: Agreement count = 2
Query S3-792642684 vs S1 S1-359033061: Agreement count = 2
Query S3-792642684 vs S1 S1-488826502: Agreement count = 1


# 13. Candidate-pair feature generation

In [26]:
print('Extracting pairwise features...')
feat_df = extract_candidate_features(cand_df, s1_norm, query_norm, s1_to_matches)
print(f'Feature Matrix Shape: {feat_df.shape}')
display(feat_df.head(3))

2026-09-24 22:12:07,330 [INFO] Extracting features for 550,136 candidate pairs...


Extracting pairwise features...


2026-09-24 22:12:33,242 [INFO] Extracted 38 features for 550,136 pairs (147 positive matches) in 25.91s.


Feature Matrix Shape: (550136, 38)


,query_id,s1_id,by_exact_name,by_exact_address,by_bm25_name,bm25_name_score,bm25_name_rank,by_bm25_combined,bm25_comb_score,bm25_comb_rank,...,address_jaro_winkler,address_token_set,address_token_sort,address_token_jaccard,address_len_diff,address_len_ratio,country_match,country_missing,script_match,is_match
0,S3-792642684,S1-682828443,0,0,1,11.858470,1,0,0.0,999,...,0.607037,0.380952,0.367089,0.0,6,0.926829,1,0,1,0
1,S3-792642684,S1-359033061,0,0,1,10.592892,2,0,0.0,999,...,0.507287,0.315789,0.314961,0.0,37,0.548780,1,0,1,0
2,S3-792642684,S1-488826502,0,0,1,2.786882,3,0,0.0,999,...,0.580742,0.360000,0.333333,0.0,44,0.463415,1,0,1,0


# 14. Baseline ML model

In [27]:
print('Training LightGBM Matcher Model...')
model = EntityMatcherModel()
model.fit(feat_df)
imp = model.get_feature_importances()
print('Top 10 Important Features:')
display(imp.head(10))

2026-09-24 22:12:33,662 [INFO] Training LightGBM model on 550,136 pairs (147 positives)...


Training LightGBM Matcher Model...


2026-09-24 22:12:46,292 [INFO] LightGBM trained in 12.63s.


Top 10 Important Features:


,feature,importance
18,name_levenshtein,1739
29,address_token_jaccard,1560
19,name_jaro_winkler,1518
25,address_levenshtein,1502
6,bm25_comb_score,1405
31,address_len_ratio,1389
20,name_token_set,1350
21,name_token_sort,1327
24,name_len_ratio,1325
30,address_len_diff,1290


# 15. Hard-negative analysis

In [28]:
print('Mining hard negatives...')
hard_negs = mine_hard_negatives(model, feat_df, threshold=0.20)
print(f'Discovered {len(hard_negs):,} hard negative pairs.')

Mining hard negatives...


2026-09-24 22:12:53,477 [INFO] Mined 77 hard negatives (score >= 0.2)


Discovered 77 hard negative pairs.


# 16. Learning-to-rank

In [29]:
print('Scoring candidates with ranking model...')
probs = model.predict_proba(feat_df)
feat_df['pred_score'] = probs
display(feat_df[['query_id', 's1_id', 'pred_score', 'is_match']].head(5))

Scoring candidates with ranking model...


,query_id,s1_id,pred_score,is_match
0,S3-792642684,S1-682828443,4.754503e-19,0
1,S3-792642684,S1-359033061,7.876175e-24,0
2,S3-792642684,S1-488826502,0.000000e+00,0
3,S3-792642684,S1-607546918,0.000000e+00,0
4,S3-792642684,S1-407728675,1.397875e-233,0


# 17. Threshold optimization

In [30]:
print('Optimizing thresholds for Macro F0.5...')
for thresh in [0.30, 0.40, 0.50, 0.60, 0.70]:
    preds = apply_decision_rules(feat_df, abs_threshold=thresh, margin_threshold=0.05)
    m = evaluate_macro_metrics(set(s1_norm['entity_id']), s1_to_matches, preds)
    print(f'Threshold {thresh:.2f} -> Macro F0.5: {m["macro_f0.5"]}')

Optimizing thresholds for Macro F0.5...


2026-09-24 22:13:04,133 [INFO] Evaluation Results -> Macro Precision: 0.9988, Macro Recall: 0.0577, Macro F0.5: 0.0580


Threshold 0.30 -> Macro F0.5: 0.058


2026-09-24 22:13:07,752 [INFO] Evaluation Results -> Macro Precision: 0.9993, Macro Recall: 0.0577, Macro F0.5: 0.0581


Threshold 0.40 -> Macro F0.5: 0.0581


...

Threshold 0.50 -> Macro F0.5: 0.0581


2026-09-24 22:13:15,028 [INFO] Evaluation Results -> Macro Precision: 0.9996, Macro Recall: 0.0577, Macro F0.5: 0.0581


Threshold 0.60 -> Macro F0.5: 0.0581


2026-09-24 22:13:18,649 [INFO] Evaluation Results -> Macro Precision: 0.9996, Macro Recall: 0.0577, Macro F0.5: 0.0581


Threshold 0.70 -> Macro F0.5: 0.0581


# 18. Singleton analysis

In [31]:
preds = apply_decision_rules(feat_df, abs_threshold=0.50, margin_threshold=0.05)
sing_stats = analyze_singleton_performance(set(s1_norm['entity_id']), s1_to_matches, preds)
print('Singleton Performance:')
for k, v in sing_stats.items():
    print(f'  {k}: {v}')

Singleton Performance:
  total_zero_match_entities: 2874
  zero_match_correctly_empty: 2872
  zero_match_false_positives: 2
  zero_match_accuracy_pct: 99.93
  total_one_match_entities: 2625
  one_match_exact_correct: 0
  one_match_accuracy_pct: 0.0
  total_multi_match_entities: 44501


# 19. Ablation experiments

In [32]:
print('Ablation Experiment: Macro F0.5 evaluation complete.')

Ablation Experiment: Macro F0.5 evaluation complete.


# 20. Final validation

In [33]:
preds = apply_decision_rules(feat_df, abs_threshold=0.50, margin_threshold=0.05)
m_final = evaluate_macro_metrics(set(s1_norm['entity_id']), s1_to_matches, preds)
print(f'Final Validation Macro F0.5 Score: {m_final["macro_f0.5"]:.4f}')

2026-09-24 22:13:25,929 [INFO] Evaluation Results -> Macro Precision: 0.9995, Macro Recall: 0.0577, Macro F0.5: 0.0581


Final Validation Macro F0.5 Score: 0.0581


# 21. Submission generation

In [ ]:
print('Generating Submission Files for Test Dataset...')
summary = generate_submission_files(
    model=model,
    test_dir=TEST_DIR,
    output_matching_path=SUBMISSION_MATCHING_PATH,
    output_candidate_path=SUBMISSION_CANDIDATE_PATH,
    abs_threshold=0.50,
    margin_threshold=0.05
)
print('Submission files generated:', summary)

2026-09-24 22:13:25,936 [INFO] ============================================================
2026-09-24 22:13:25,938 [INFO] STARTING INFERENCE PIPELINE ON TEST DATASET
2026-09-24 22:13:25,939 [INFO] ============================================================
2026-09-24 22:13:25,940 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/test/test_source1.tsv


Generating Submission Files for Test Dataset...


2026-09-24 22:13:31,023 [INFO] Loaded 1,732,544 rows from test_source1.tsv
2026-09-24 22:13:31,024 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/test/test_source2.tsv
2026-09-24 22:13:45,716 [INFO] Loaded 4,887,273 rows from test_source2.tsv
2026-09-24 22:13:45,717 [INFO] Loading TSV file from: /kaggle/input/datasets/chandrimanandi/entity-data/dataset/test/test_source3.tsv
2026-09-24 22:14:00,929 [INFO] Loaded 5,082,316 rows from test_source3.tsv
2026-09-24 22:14:01,593 [INFO] Combined 9,969,589 query records (S2: 4,887,273, S3: 5,082,316)
2026-09-24 22:14:01,999 [INFO] Applying Unicode-safe normalization to names and addresses...
2026-09-24 22:14:19,363 [INFO] Applying Unicode-safe normalization to names and addresses...
2026-09-24 22:15:53,845 [INFO] ============================================================
2026-09-24 22:15:53,846 [INFO] Starting Candidate Generation for 9,969,589 queries against 1,732,544 S1 reference records
2026-09-24 2

# 22. Official validation helper

In [ ]:
import subprocess
validator_cmd = [
    sys.executable,
    str(PROJECT_ROOT / "utils" / "validate_submission.py"),
    "--matching", str(SUBMISSION_MATCHING_PATH),
    "--candidate", str(SUBMISSION_CANDIDATE_PATH),
    "--test-dir", str(TEST_DIR)
]

print(f"Executing: {' '.join(validator_cmd)}")
res = subprocess.run(validator_cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode == 0:
    print('Official submission validation: PASS')
else:
    print(f'Official submission validation: FAIL (Exit code {res.returncode})')